# CPU Tests — RSPrompter & DSMPrompter

Tests **sans GPU, sans checkpoint SAM/SAM2**. Vérifie la logique des modèles avec des données dummy.

**Durée totale : ~30 secondes**

```bash
conda activate seg-test
jupyter notebook notebooks/cpu_tests.ipynb
```

## 0. Imports & dépendances

In [ ]:
import os, sys, json, tempfile
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from scipy.optimize import linear_sum_assignment

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : CPU (forced)")

PASS = "  PASS"
FAIL = "  FAIL"
SKIP = "  SKIP"

In [ ]:
# Dépendances optionnelles (pas bloquantes pour les tests ci-dessous)
print("Dépendances installées :")
for pkg in ["segment_anything", "sam2", "detectron2", "cv2", "pycocotools", "scipy"]:
    try:
        __import__(pkg)
        print(f"  OK      {pkg}")
    except ImportError:
        print(f"  MISSING {pkg}  (non requis pour ces tests CPU)")

---
## 1. Composants RSPrompter
Recopiés inline — aucune dépendance externe requise.

In [ ]:
# ---- Copie minimale des classes RSPrompter (sans SAM) ----

class PromptEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim, 3, 1, 1),
            nn.LayerNorm([embed_dim, 64, 64]), nn.GELU(),
            nn.Conv2d(embed_dim, embed_dim, 3, 1, 1),
            nn.LayerNorm([embed_dim, 64, 64]), nn.GELU(),
        )
        self.box_head   = nn.Conv2d(embed_dim, 4, 1)
        self.score_head = nn.Conv2d(embed_dim, 1, 1)
    def forward(self, x):
        x = self.conv_layers(x)
        return self.box_head(x), self.score_head(x)

class HungarianMatcher(nn.Module):
    def __init__(self, cost_class=1.0, cost_bbox=5.0):
        super().__init__()
        self.cost_class, self.cost_bbox = cost_class, cost_bbox
    @torch.no_grad()
    def forward(self, outputs, targets):
        pred_logits, pred_boxes = outputs["pred_logits"], outputs["pred_boxes"]
        indices = []
        for b in range(pred_logits.shape[0]):
            tgt_boxes = targets[b]["boxes"]
            if tgt_boxes.shape[0] == 0:
                indices.append((torch.tensor([], dtype=torch.int64), torch.tensor([], dtype=torch.int64)))
                continue
            prob = pred_logits[b].sigmoid().squeeze(-1)
            cost = -prob.unsqueeze(1) * self.cost_class + torch.cdist(pred_boxes[b], tgt_boxes, p=1) * self.cost_bbox
            row, col = linear_sum_assignment(cost.cpu())
            indices.append((torch.as_tensor(row, dtype=torch.int64), torch.as_tensor(col, dtype=torch.int64)))
        return indices

class RSPrompterLoss(nn.Module):
    def __init__(self, matcher):
        super().__init__()
        self.matcher = matcher
    def forward(self, outputs, targets):
        indices   = self.matcher(outputs, targets)
        batch_idx = torch.cat([torch.full_like(s, i) for i, (s, _) in enumerate(indices)])
        src_idx   = torch.cat([s for s, _ in indices])
        target_classes = torch.zeros_like(outputs["pred_logits"])
        if batch_idx.numel() > 0:
            target_classes[batch_idx, src_idx, 0] = 1.0
        loss_class = F.binary_cross_entropy_with_logits(outputs["pred_logits"], target_classes)
        loss_bbox, loss_mask = torch.tensor(0.0), torch.tensor(0.0)
        if batch_idx.numel() > 0:
            loss_bbox = F.l1_loss(
                outputs["pred_boxes"][batch_idx, src_idx],
                torch.cat([t["boxes"][tgt] for t, (_, tgt) in zip(targets, indices)])
            )
            src_masks = outputs["pred_masks"][batch_idx, src_idx]
            tgt_masks = torch.cat([t["masks"][tgt] for t, (_, tgt) in zip(targets, indices)])
            if src_masks.shape[-2:] != tgt_masks.shape[-2:]:
                src_masks = F.interpolate(src_masks, size=tgt_masks.shape[-2:], mode="bilinear", align_corners=False)
            loss_mask = F.binary_cross_entropy_with_logits(src_masks.squeeze(1), tgt_masks.float())
        return 2.0 * loss_class + 5.0 * loss_bbox + 5.0 * loss_mask

print("Classes RSPrompter définies (inline, sans dépendance SAM).")

In [ ]:
# Test 1.1 — PromptEncoder : shapes
enc = PromptEncoder(embed_dim=256)
x   = torch.randn(2, 256, 64, 64)
box_out, score_out = enc(x)
assert box_out.shape   == (2, 4, 64, 64), box_out.shape
assert score_out.shape == (2, 1, 64, 64), score_out.shape
print(PASS, f"PromptEncoder  box={tuple(box_out.shape)}  score={tuple(score_out.shape)}")

In [ ]:
# Test 1.2 — HungarianMatcher : nombre de paires
B, N = 2, 10
outputs = {
    "pred_logits": torch.randn(B, N, 1),
    "pred_boxes":  torch.rand(B, N, 4) * 1024,
    "pred_masks":  torch.randn(B, N, 1, 256, 256),
}
targets = [
    {"boxes": torch.rand(3, 4)*1024, "masks": torch.zeros(3, 1024, 1024)},
    {"boxes": torch.rand(5, 4)*1024, "masks": torch.zeros(5, 1024, 1024)},
]
matcher = HungarianMatcher()
indices = matcher(outputs, targets)
assert len(indices[0][0]) <= 3,  "batch 0 : trop de paires"
assert len(indices[1][0]) <= 5,  "batch 1 : trop de paires"
print(PASS, f"HungarianMatcher  batch0={len(indices[0][0])} paires / 3 GT   batch1={len(indices[1][0])} paires / 5 GT")

In [ ]:
# Test 1.3 — RSPrompterLoss : scalaire > 0, pas NaN
criterion = RSPrompterLoss(HungarianMatcher())
loss = criterion(outputs, targets)
assert loss.ndim == 0,           "pas un scalaire"
assert loss.item() == loss.item(), "NaN!"
assert loss.item() > 0
print(PASS, f"RSPrompterLoss  loss={loss.item():.4f}")

In [ ]:
# Test 1.4 — RSPrompterLoss avec batch vide (0 GT)
targets_empty = [
    {"boxes": torch.zeros(0, 4), "masks": torch.zeros(0, 1024, 1024)},
    {"boxes": torch.zeros(0, 4), "masks": torch.zeros(0, 1024, 1024)},
]
loss_empty = criterion(outputs, targets_empty)
assert loss_empty.item() == loss_empty.item(), "NaN avec 0 GT!"
print(PASS, f"RSPrompterLoss (0 GT)  loss={loss_empty.item():.4f}")

In [ ]:
# Test 1.5 — decode_boxes : pas de NaN/Inf avec offsets normaux
def decode_boxes_rsprompter(box_offsets, scores, num_proposals=10):
    B, _, H, W = box_offsets.shape
    stride = 1024.0 / H
    y_grid, x_grid = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    cx = (x_grid * stride + stride / 2).flatten()
    cy = (y_grid * stride + stride / 2).flatten()
    offsets    = box_offsets.permute(0, 2, 3, 1).reshape(B, -1, 4)
    flat_scores = scores.view(B, -1)
    scale = stride * 4.0
    dx, dy = offsets[..., 0]*scale, offsets[..., 1]*scale
    w  = torch.exp(offsets[..., 2].clamp(-10, 10)) * scale  # clamp RSPrompter
    h  = torch.exp(offsets[..., 3].clamp(-10, 10)) * scale
    boxes = torch.stack([
        (cx.unsqueeze(0)+dx)-w/2, (cy.unsqueeze(0)+dy)-h/2,
        (cx.unsqueeze(0)+dx)+w/2, (cy.unsqueeze(0)+dy)+h/2
    ], dim=-1).clamp(0, 1024)
    k = min(num_proposals, flat_scores.shape[1])
    _, topk_idx = torch.topk(flat_scores, k=k, dim=1)
    batch_idx = torch.arange(B)[:, None]
    return boxes[batch_idx, topk_idx], flat_scores[batch_idx, topk_idx]

dummy_offsets = torch.randn(2, 4, 64, 64)
dummy_scores  = torch.randn(2, 1, 64, 64)
boxes_out, scores_out = decode_boxes_rsprompter(dummy_offsets, dummy_scores, num_proposals=10)
assert torch.isfinite(boxes_out).all(),  "NaN/Inf dans les boxes"
assert boxes_out.shape  == (2, 10, 4)
assert scores_out.shape == (2, 10)
print(PASS, f"decode_boxes RSPrompter  boxes={tuple(boxes_out.shape)}, toutes finies")

In [ ]:
# Test 1.6 — decode_boxes DSMPrompter : overflow sans clamp
def decode_boxes_dsm(box_offsets, scores, num_proposals=10):
    B, _, H, W = box_offsets.shape
    stride = 1024.0 / H
    y_grid, x_grid = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    cx = (x_grid * stride + stride / 2).flatten()
    cy = (y_grid * stride + stride / 2).flatten()
    offsets    = box_offsets.permute(0, 2, 3, 1).reshape(B, -1, 4)
    flat_scores = scores.view(B, -1)
    scale = stride * 4.0
    dx, dy = offsets[..., 0]*scale, offsets[..., 1]*scale
    w  = torch.exp(offsets[..., 2]) * scale  # PAS de clamp — bug DSMPrompter
    h  = torch.exp(offsets[..., 3]) * scale
    boxes = torch.stack([
        (cx.unsqueeze(0)+dx)-w/2, (cy.unsqueeze(0)+dy)-h/2,
        (cx.unsqueeze(0)+dx)+w/2, (cy.unsqueeze(0)+dy)+h/2
    ], dim=-1).clamp(0, 1024)
    k = min(num_proposals, flat_scores.shape[1])
    _, topk_idx = torch.topk(flat_scores, k=k, dim=1)
    batch_idx = torch.arange(B)[:, None]
    return boxes[batch_idx, topk_idx], flat_scores[batch_idx, topk_idx]

# Offsets normaux → OK même sans clamp
boxes_dsm, _ = decode_boxes_dsm(dummy_offsets, dummy_scores)
print(PASS, f"decode_boxes DSMPrompter (offsets normaux)  boxes={tuple(boxes_dsm.shape)}")

# Offsets extrêmes → Inf sans clamp
extreme_offsets = torch.full((2, 4, 64, 64), 50.0)  # exp(50*scale) → Inf
boxes_extreme, _ = decode_boxes_dsm(extreme_offsets, dummy_scores)
has_inf = not torch.isfinite(boxes_extreme).all()
if has_inf:
    print("  WARN  decode_boxes DSMPrompter : NaN/Inf avec offsets extrêmes (manque .clamp(-10,10))")
    print("         → Risque de loss=NaN au début de l'entraînement si le réseau diverge")
else:
    print(PASS, "decode_boxes DSMPrompter : OK même avec offsets extrêmes")

In [ ]:
# Test 1.7 — EmbeddingDataset avec fichiers .pt dummy
from torch.utils.data import DataLoader

class EmbeddingDataset:
    """Copie minimale de RSPrompter.EmbeddingDataset."""
    def __init__(self, json_file, embed_dir):
        with open(json_file) as f:
            self.coco = json.load(f)
        self.embed_dir = embed_dir
        self.images    = {img['id']: img for img in self.coco['images']}
        self.annotations = {}
        for ann in self.coco['annotations']:
            self.annotations.setdefault(ann['image_id'], []).append(ann)
        self.img_ids   = list(self.images.keys())
        self.orig_sizes = {img['id']: (img.get('height', 1024), img.get('width', 1024))
                           for img in self.coco['images']}
    def __len__(self): return len(self.img_ids)
    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        path   = os.path.join(self.embed_dir, f"{img_id}.pt")
        emb    = torch.load(path) if os.path.exists(path) else torch.zeros(256, 64, 64)
        anns   = self.annotations.get(img_id, [])
        boxes  = torch.tensor([[a['bbox'][0], a['bbox'][1],
                                 a['bbox'][0]+a['bbox'][2], a['bbox'][1]+a['bbox'][3]]
                                for a in anns], dtype=torch.float32) if anns else torch.zeros(0,4)
        masks  = torch.zeros(len(anns), 1024, 1024)
        return {"embedding": emb, "targets": {"boxes": boxes, "masks": masks}}

def collate_fn(batch):
    return {"embedding": torch.stack([b['embedding'] for b in batch]),
            "targets":   [b['targets'] for b in batch]}

with tempfile.TemporaryDirectory() as tmpdir:
    embed_dir = os.path.join(tmpdir, "emb")
    os.makedirs(embed_dir)
    fake_coco = {
        "images":      [{"id": i, "file_name": f"{i}.png", "height": 1024, "width": 1024} for i in range(1, 6)],
        "annotations": [{"id": 1, "image_id": 1, "category_id": 1, "bbox": [100,100,200,150], "segmentation": []}],
        "categories":  [{"id": i, "name": f"tree_{i}"} for i in range(1,8)]
    }
    json_path = os.path.join(tmpdir, "train.json")
    with open(json_path, "w") as f: json.dump(fake_coco, f)
    for i in range(1, 6):
        torch.save(torch.randn(256, 64, 64), os.path.join(embed_dir, f"{i}.pt"))

    ds     = EmbeddingDataset(json_path, embed_dir)
    loader = DataLoader(ds, batch_size=2, collate_fn=collate_fn)
    batch  = next(iter(loader))

    assert batch["embedding"].shape == (2, 256, 64, 64)
    assert len(batch["targets"]) == 2

print(PASS, f"EmbeddingDataset  len=5, batch embedding={tuple(batch['embedding'].shape)}")

---
## 2. Composants DSMPrompter

In [ ]:
# ---- Copie minimale des classes DSMPrompter (sans SAM2) ----

class DSMEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, 2, 1),  nn.LayerNorm([16, 512, 512]),  nn.GELU(),
            nn.Conv2d(16, 32, 3, 2, 1), nn.LayerNorm([32, 256, 256]),  nn.GELU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.LayerNorm([64, 128, 128]),  nn.GELU(),
            nn.Conv2d(64, embed_dim, 3, 2, 1), nn.LayerNorm([embed_dim, 64, 64]), nn.GELU(),
            nn.Conv2d(embed_dim, embed_dim, 1)
        )
    def forward(self, x): return self.net(x)

class PromptProposalHead(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.conv_box   = nn.Conv2d(embed_dim, 4, 1)
        self.conv_score = nn.Conv2d(embed_dim, 1, 1)
    def forward(self, x): return self.conv_box(x), self.conv_score(x)

class DSMLoss(nn.Module):
    def __init__(self, matcher):
        super().__init__()
        self.matcher = matcher
    def forward(self, outputs, targets):
        indices   = self.matcher(outputs, targets)
        batch_idx = torch.cat([torch.full_like(s, i) for i, (s, _) in enumerate(indices)])
        src_idx   = torch.cat([s for s, _ in indices])
        target_classes = torch.zeros_like(outputs["pred_logits"])
        if batch_idx.numel() > 0:
            target_classes[batch_idx, src_idx, 0] = 1.0
        loss_class = F.binary_cross_entropy_with_logits(outputs["pred_logits"], target_classes)
        loss_bbox, loss_mask = torch.tensor(0.0), torch.tensor(0.0)
        if batch_idx.numel() > 0:
            loss_bbox = F.l1_loss(
                outputs["pred_boxes"][batch_idx, src_idx],
                torch.cat([t["boxes"][tgt] for t, (_, tgt) in zip(targets, indices)])
            )
            src_masks = outputs["pred_masks"][batch_idx, src_idx]
            tgt_masks = torch.cat([t["masks"][tgt] for t, (_, tgt) in zip(targets, indices)])
            if src_masks.shape[-2:] != tgt_masks.shape[-2:]:
                src_masks = F.interpolate(src_masks, size=tgt_masks.shape[-2:], mode="bilinear", align_corners=False)
            loss_mask = F.binary_cross_entropy_with_logits(src_masks.squeeze(1), tgt_masks.float())
        return 2.0*loss_class + 5.0*loss_bbox + 5.0*loss_mask

print("Classes DSMPrompter définies (inline, sans dépendance SAM2).")

In [ ]:
# Test 2.1 — DSMEncoder : (B, 1, 1024, 1024) → (B, 256, 64, 64)
enc_dsm = DSMEncoder(embed_dim=256)
dsm_in  = torch.randn(2, 1, 1024, 1024)
dsm_out = enc_dsm(dsm_in)
assert dsm_out.shape == (2, 256, 64, 64), dsm_out.shape
params = sum(p.numel() for p in enc_dsm.parameters())
print(PASS, f"DSMEncoder  {tuple(dsm_in.shape)} → {tuple(dsm_out.shape)}  ({params:,} params)")

In [ ]:
# Test 2.2 — Fusion image + DSM embedding
img_emb  = torch.randn(2, 256, 64, 64)  # SAM2 backbone output (simulé)
fused    = img_emb + dsm_out
assert fused.shape == (2, 256, 64, 64)
assert torch.isfinite(fused).all()
print(PASS, f"Fusion img_emb + dsm_emb  shape={tuple(fused.shape)}")

In [ ]:
# Test 2.3 — PromptProposalHead shapes
head      = PromptProposalHead(embed_dim=256)
box_off, logits = head(fused)
assert box_off.shape == (2, 4, 64, 64), box_off.shape
assert logits.shape  == (2, 1, 64, 64), logits.shape
print(PASS, f"PromptProposalHead  box_offsets={tuple(box_off.shape)}  logits={tuple(logits.shape)}")

In [ ]:
# Test 2.4 — DSMLoss : scalaire > 0, pas NaN
B, K = 2, 10
fake_outputs = {
    "pred_logits": torch.randn(B, K, 1),
    "pred_boxes":  torch.rand(B, K, 4) * 1024,
    "pred_masks":  torch.randn(B, K, 1, 256, 256),
}
fake_targets = [
    {"boxes": torch.rand(4, 4)*1024, "masks": torch.zeros(4, 1024, 1024)},
    {"boxes": torch.rand(2, 4)*1024, "masks": torch.zeros(2, 1024, 1024)},
]
loss_dsm = DSMLoss(HungarianMatcher())(fake_outputs, fake_targets)
assert loss_dsm.ndim == 0
assert loss_dsm.item() == loss_dsm.item(), "NaN!"
assert loss_dsm.item() > 0
print(PASS, f"DSMLoss  loss={loss_dsm.item():.4f}")

In [ ]:
# Test 2.5 — Gradient flow : les paramètres entraînables reçoivent bien un gradient
enc_g  = DSMEncoder()
head_g = PromptProposalHead()

dsm_input = torch.randn(1, 1, 1024, 1024)
img_input = torch.randn(1, 256, 64, 64)

fused_g      = img_input + enc_g(dsm_input)
box_g, log_g = head_g(fused_g)

# Loss proxy
loss_g = box_g.mean() + log_g.mean()
loss_g.backward()

no_grad = [n for n, p in list(enc_g.named_parameters()) + list(head_g.named_parameters())
           if p.grad is None]
assert len(no_grad) == 0, f"Pas de gradient pour : {no_grad}"
print(PASS, f"Gradient flow OK — tous les {sum(1 for _ in list(enc_g.parameters()) + list(head_g.parameters()))} paramètres reçoivent un gradient")

---
## 3. Vérification des données

In [ ]:
# Chemins relatifs depuis la racine du projet
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_ROOT    = os.path.abspath(os.path.join(PROJECT_ROOT, "../../../Output"))

print(f"Project root : {PROJECT_ROOT}")
print(f"Data root    : {DATA_ROOT}")
print()

splits = ["train", "val", "test"]
print("COCO JSONs :")
for split in splits:
    path = os.path.join(DATA_ROOT, "coco", f"{split}.json")
    if os.path.exists(path):
        with open(path) as f: d = json.load(f)
        n_img = len(d.get("images", []))
        n_ann = len(d.get("annotations", []))
        n_cat = len(d.get("categories", []))
        avg   = round(n_ann / n_img, 1) if n_img else 0
        print(f"  OK    {split:5s}  {n_img:4d} images  {n_ann:5d} annotations  {n_cat} catégories  ({avg} ann/img)")
    else:
        print(f"  MISS  {split:5s}  {path}")

print()
print("Images :")
for split in splits:
    for modality in ["RGB", "DSM"]:
        d = os.path.join(DATA_ROOT, split, modality)
        if os.path.isdir(d):
            n = len(os.listdir(d))
            print(f"  OK    {split}/{modality:<4}  {n} fichiers")
        else:
            print(f"  MISS  {split}/{modality}")

In [ ]:
# Vérification des checkpoints existants (pas requis pour ces tests)
BASE_MODELS = os.path.join(PROJECT_ROOT, "src", "models")
checkpoints = {
    "SAM ViT-H":           os.path.abspath(os.path.join(BASE_MODELS, "../../../../sam_vit_h_4b8939.pth")),
    "SAM2 large":          os.path.abspath(os.path.join(BASE_MODELS, "../../../../checkpoints/sam2.1_hiera_large.pt")),
    "Mask R-CNN RGB":      os.path.abspath(os.path.join(BASE_MODELS, "../../../../Résulats post training/output_mask_rcnn_RGB/model_final.pth")),
    "Mask R-CNN RGB+DSM":  os.path.abspath(os.path.join(BASE_MODELS, "../../../../Résulats post training/Output_mask_rcnn_RGB_DSM/model_final.pth")),
    "Faster R-CNN":        os.path.abspath(os.path.join(BASE_MODELS, "../../../../Résulats post training/output_faster_rcnn/model_final.pth")),
    "SAM embeddings":      os.path.join(BASE_MODELS, "sam_embeddings_vith", "train"),
}
print("Checkpoints (requis pour l'entraînement sur GPU) :")
for label, path in checkpoints.items():
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path)/1e9:.1f} GB" if exists and os.path.isfile(path) else ""
    print(f"  {'OK  ' if exists else 'MISS'}  {label:<22}  {size}")

---
## 4. Résumé

In [ ]:
print("=" * 55)
print("RÉSUMÉ")
print("=" * 55)
print("Tests logique (CPU, sans checkpoint) : tous ci-dessus")
print()
print("Pour passer à l'entraînement sur GPU :")
sam_ok  = os.path.exists(os.path.abspath(os.path.join(BASE_MODELS, "../../../../sam_vit_h_4b8939.pth")))
sam2_ok = os.path.exists(os.path.abspath(os.path.join(BASE_MODELS, "../../../../checkpoints/sam2.1_hiera_large.pt")))
emb_ok  = os.path.isdir(os.path.join(BASE_MODELS, "sam_embeddings_vith", "train"))

step = 1
if not sam_ok:
    print(f"  {step}. Télécharger SAM ViT-H (2.4 GB)")
    print("     wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth")
    step += 1
if not sam2_ok:
    print(f"  {step}. Télécharger SAM2 large (900 MB)")
    print("     mkdir -p checkpoints && wget -P checkpoints https://dl.fbaipublicfiles.com/segment_anything/2.1/sam2.1_hiera_large.pt")
    step += 1
if sam_ok and not emb_ok:
    print(f"  {step}. Pré-calculer les embeddings SAM (étape 1 RSPrompter)")
    print("     python src/misc/Pre_Compute_SAM.py")
    step += 1
if sam_ok and emb_ok:
    print(f"  {step}. Lancer RSPrompter")
    print("     python src/models/RSPrompter.py")
    step += 1
if sam2_ok:
    print(f"  {step}. Lancer DSMPrompter")
    print("     python src/models/DSM_Prompter.py")
print()